# Experiment 5: Consolidate Results & Generate Publication Figures

**Purpose:** Collect all results from experiments 01–04, generate manuscript-ready tables and figures.

**Requires:** Run after completing notebooks 01–04. Upload `results.zip` files if running on a fresh session.

**Outputs:**
- `manuscript_table1_voc.csv` — VOC results
- `manuscript_table2_coco.csv` — COCO results
- `manuscript_table3_sota.csv` — SOTA comparison
- `publication_figures.png` — Combined figure panel
- `pareto_front.png` — Accuracy vs params plot

In [ ]:
!rm -rf /content/tinyYOLO 2>/dev/null
!git clone https://github.com/ShMazumder/tinyYOLO.git /content/tinyYOLO
%cd /content/tinyYOLO
!pip install -e . -q

import json, glob, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

matplotlib.rcParams.update({'font.size': 11, 'font.family': 'sans-serif'})
RESULTS = Path('experiments/results')
RESULTS.mkdir(parents=True, exist_ok=True)
print('Ready. Upload previous results zips if needed.')

In [ ]:
# === Upload previous results (if on fresh session) ===
# Uncomment to upload:
# from google.colab import files
# uploaded = files.upload()  # Upload voc_5seed_results.zip, coco_results.zip, etc.
# for name in uploaded:
#     !unzip -o {name} -d experiments/

# List available results
exps = sorted([d.name for d in RESULTS.iterdir() if d.is_dir()])
print(f'Found {len(exps)} experiments:')
for e in exps:
    print(f'  • {e}')

In [ ]:
# === Table 1: VOC Results (mean ± std) ===
print('=' * 80)
print('  TABLE 1: Pascal VOC 2007+2012 (416×416, 300 epochs)')
print('=' * 80)
print(f'{"Model":<20} {"Params":>8} {"mAP@50":>16} {"mAP@50-95":>16} {"P":>12} {"R":>12}')
print('-' * 80)

rows = []
for variant, label in [('q', 'TinyYOLO-q'), ('std', 'TinyYOLO-std')]:
    m50, m95, ps, rs = [], [], [], []
    for f in sorted(glob.glob(f'{RESULTS}/voc-{variant}-416-seed*/config.json')):
        with open(f) as fh:
            cfg = json.load(fh)
        fm = cfg.get('final_metrics', {})
        m50.append(fm.get('mAP50', 0))
        m95.append(fm.get('mAP50_95', fm.get('mAP50-95', 0)))
        ps.append(fm.get('precision', 0))
        rs.append(fm.get('recall', 0))
    if m50:
        params = cfg.get('params_M', '?')
        row = [label, f'{params}M',
               f'{np.mean(m50)*100:.1f}±{np.std(m50)*100:.1f}',
               f'{np.mean(m95)*100:.1f}±{np.std(m95)*100:.1f}',
               f'{np.mean(ps)*100:.1f}±{np.std(ps)*100:.1f}',
               f'{np.mean(rs)*100:.1f}±{np.std(rs)*100:.1f}']
        rows.append(row)
        print(f'{row[0]:<20} {row[1]:>8} {row[2]:>16} {row[3]:>16} {row[4]:>12} {row[5]:>12}')

# Save CSV
import csv
with open(RESULTS / 'manuscript_table1_voc.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['Model', 'Params', 'mAP@50', 'mAP@50-95', 'Precision', 'Recall'])
    w.writerows(rows)
print(f'\nSaved: {RESULTS}/manuscript_table1_voc.csv')

In [ ]:
# === Table 2: COCO Results ===
print('=' * 80)
print('  TABLE 2: COCO val2017 (416×416, 300 epochs)')
print('=' * 80)

rows = []
for variant, label in [('q', 'TinyYOLO-q'), ('std', 'TinyYOLO-std')]:
    m50, m95 = [], []
    for f in sorted(glob.glob(f'{RESULTS}/coco*-{variant}-416-seed*/config.json')):
        with open(f) as fh:
            cfg = json.load(fh)
        fm = cfg.get('final_metrics', {})
        m50.append(fm.get('mAP50', 0))
        m95.append(fm.get('mAP50_95', fm.get('mAP50-95', 0)))
    if m50:
        row = [label, cfg.get('params_M', '?'),
               f'{np.mean(m50)*100:.1f}±{np.std(m50)*100:.1f}',
               f'{np.mean(m95)*100:.1f}±{np.std(m95)*100:.1f}']
        rows.append(row)
        print(f'  {row[0]}: mAP@50={row[2]}, mAP@50-95={row[3]}')

with open(RESULTS / 'manuscript_table2_coco.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['Model', 'Params', 'mAP@50', 'mAP@50-95'])
    w.writerows(rows)
print(f'Saved: {RESULTS}/manuscript_table2_coco.csv')

In [ ]:
# === Pareto Front: Params vs mAP@50 ===
fig, ax = plt.subplots(figsize=(10, 7))

# TinyYOLO results
for f in sorted(glob.glob(f'{RESULTS}/voc-*-seed42/config.json')):
    with open(f) as fh:
        cfg = json.load(fh)
    fm = cfg.get('final_metrics', {})
    params = cfg.get('params_M', 0)
    m50 = fm.get('mAP50', 0) * 100
    name = Path(f).parent.name
    color = '#2196F3' if '-q-' in name else '#FF9800'
    ax.scatter(params, m50, s=150, c=color, zorder=5, edgecolors='#333', linewidth=1.5)
    ax.annotate(name.replace('voc-','').replace('-416-seed42',''),
                (params, m50), textcoords='offset points', xytext=(8, 5), fontsize=10)

# SOTA baselines (published values)
sota = [
    ('YOLO-Fastest', 0.25, 35.1, '#E91E63'),
    ('MCUNet', 0.74, 42.8, '#E91E63'),
    ('NanoDet-m', 0.95, 48.3, '#E91E63'),
    ('PicoDet-XS', 0.93, 50.1, '#E91E63'),
    ('YOLOv5n', 1.9, 55.2, '#9C27B0'),
    ('YOLOv8n', 3.2, 63.5, '#9C27B0'),
]
for name, p, m, c in sota:
    ax.scatter(p, m, s=80, marker='D', c=c, zorder=4, alpha=0.7)
    ax.annotate(name, (p, m), textcoords='offset points', xytext=(5, -12),
                fontsize=8, color=c)

ax.set_xlabel('Parameters (M)', fontsize=13)
ax.set_ylabel('mAP@50 (%) — VOC', fontsize=13)
ax.set_title('Accuracy–Efficiency Pareto Front', fontsize=14, fontweight='bold')
ax.axvspan(0, 0.5, alpha=0.08, color='green', label='Sub-0.5M target zone')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / 'pareto_front.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: pareto_front.png')

In [ ]:
# === Combined Publication Figure (2×2 panel) ===
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel A: Training curves (load all histories)
ax = axes[0, 0]
for variant, color, label in [('q', '#2196F3', 'Quantized'), ('std', '#FF9800', 'Standard')]:
    histories = []
    for f in sorted(glob.glob(f'{RESULTS}/voc-{variant}-416-seed*/history.json')):
        with open(f) as fh:
            histories.append(json.load(fh))
    if histories:
        min_len = min(len(h) for h in histories)
        mean_loss = [np.mean([h[e].get('total', 0) for h in histories]) for e in range(min_len)]
        ax.plot(range(1, min_len+1), mean_loss, color=color, linewidth=2, label=label)
ax.set_xlabel('Epoch'); ax.set_ylabel('Total Loss')
ax.set_title('(a) Training Loss Convergence'); ax.legend(); ax.grid(alpha=0.3)

# Panel B: mAP convergence
ax = axes[0, 1]
for variant, color, label in [('q', '#2196F3', 'Quantized'), ('std', '#FF9800', 'Standard')]:
    histories = []
    for f in sorted(glob.glob(f'{RESULTS}/voc-{variant}-416-seed*/history.json')):
        with open(f) as fh:
            histories.append(json.load(fh))
    if histories:
        min_len = min(len(h) for h in histories)
        mean_map = [np.mean([h[e].get('mAP50', 0) for h in histories]) for e in range(min_len)]
        ax.plot(range(1, min_len+1), mean_map, color=color, linewidth=2, label=label)
ax.set_xlabel('Epoch'); ax.set_ylabel('mAP@50')
ax.set_title('(b) mAP@50 Progression'); ax.legend(); ax.grid(alpha=0.3)

# Panel C: Resolution ablation
ax = axes[1, 0]
res_data = []
for f in sorted(glob.glob(f'{RESULTS}/ablation-a6-*/config.json')):
    with open(f) as fh:
        cfg = json.load(fh)
    res = Path(f).parent.name.split('-')[-1]
    m50 = cfg.get('final_metrics', {}).get('mAP50', 0) * 100
    res_data.append((int(res), m50))
if res_data:
    res_data.sort()
    resolutions, maps = zip(*res_data)
    ax.bar([str(r) for r in resolutions], maps, color='#2196F3', edgecolor='#333')
    for i, m in enumerate(maps):
        ax.text(i, m + 0.3, f'{m:.1f}', ha='center', fontweight='bold')
ax.set_xlabel('Resolution'); ax.set_ylabel('mAP@50 (%)')
ax.set_title('(c) Resolution Ablation'); ax.grid(axis='y', alpha=0.3)

# Panel D: Variant comparison box plot
ax = axes[1, 1]
box_data, box_labels = [], []
for variant, label in [('q', 'Quantized'), ('std', 'Standard')]:
    maps = []
    for f in sorted(glob.glob(f'{RESULTS}/voc-{variant}-416-seed*/config.json')):
        with open(f) as fh:
            maps.append(json.load(fh).get('final_metrics', {}).get('mAP50', 0) * 100)
    if maps:
        box_data.append(maps); box_labels.append(label)
if box_data:
    bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True)
    for patch, c in zip(bp['boxes'], ['#2196F3', '#FF9800']):
        patch.set_facecolor(c); patch.set_alpha(0.7)
ax.set_ylabel('mAP@50 (%)'); ax.set_title('(d) Variant Comparison (5 seeds)')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('TinyYOLO — Publication Results Summary', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'publication_figures.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: publication_figures.png')

In [ ]:
# === Full Summary — All Experiments ===
print('=' * 90)
print(f'{"Experiment":<40} {"Params":>7} {"mAP@50":>8} {"mAP50-95":>9} {"P":>7} {"R":>7}')
print('-' * 90)

all_summary = []
for f in sorted(RESULTS.rglob('config.json')):
    with open(f) as fh:
        cfg = json.load(fh)
    fm = cfg.get('final_metrics', {})
    name = f.parent.name
    row = {
        'name': name, 'params': cfg.get('params_M', 0),
        'mAP50': fm.get('mAP50', 0), 'mAP50_95': fm.get('mAP50_95', fm.get('mAP50-95', 0)),
        'precision': fm.get('precision', 0), 'recall': fm.get('recall', 0)
    }
    all_summary.append(row)
    print(f'{name:<40} {row["params"]:>6.2f}M {row["mAP50"]:>8.4f} {row["mAP50_95"]:>9.4f} '
          f'{row["precision"]:>7.4f} {row["recall"]:>7.4f}')

with open(RESULTS / 'all_experiments_summary.json', 'w') as f:
    json.dump(all_summary, f, indent=2)
print(f'\nTotal: {len(all_summary)} experiments')
print(f'Saved: {RESULTS}/all_experiments_summary.json')

In [ ]:
# === Download Everything ===
!cd experiments && zip -r /content/all_results_final.zip results/
print('\n📥 Download: /content/all_results_final.zip')
print('Contains all CSVs, PNGs, JSONs, and model checkpoints.')